### Tasks

* 1. Build a simple movie recommendation neural network using Keras Functional API that takes two inputs: user age and favorite genre (as integers), and outputs a predicted movie rating between 1 and 5.

In [175]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model

In [152]:
user_age_input = Input(shape=(1,), name='user_age')
genre_input = Input(shape=(1,), name='favorite_genre_id')

In [153]:
genre_embedded = Embedding(input_dim=20, output_dim=8, name='genre_embedding')(genre_input)
genre_flat = Flatten()(genre_embedded)

In [154]:
x = Concatenate(name='concat_features')([user_age_input, genre_flat])

In [155]:
x = Dense(32, activation='relu')(x)
x = Dense(16, activation='relu')(x)
raw_output = Dense(1, activation='sigmoid')(x)

In [156]:
rating_output = tf.keras.layers.Lambda(lambda X: 1.0 + 4.0 * x, name='predicted_rating')(raw_output)

In [157]:
movie_model = Model(inputs=[user_age_input, genre_input], outputs=rating_output, name='movie_recommender')
movie_model.summary()

Model: "movie_recommender"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ favorite_genre_id   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_embedding     │ (None, 1, 8)      │        160 │ favorite_genre_i… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_age            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 8)         │          0 │ genre_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_features     │ (None, 9)         │          0 │ user_age[0][0],   │
│ (Concatenate)       │                   │            │ flatten_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 32)        │        320 │ concat_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 16)        │        528 │ dense_45[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_47 (Dense)    │ (None, 1)         │         17 │ dense_46[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ predicted_rating    │ (None, 16)        │          0 │ dense_47[0][0]    │
│ (Lambda)            │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,025 (4.00 KB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 0 (0.00 B)

* 2. Create a multi-output neural network using Keras Functional API that predicts both the delivery time (in minutes) and delivery rating (1-5 stars) for a Zomato-style food order, given order amount and distance as inputs.<br><br><em><strong>Hint:</strong> Use two Dense output layers, one for each prediction.</em>

In [158]:
order_amount = Input(shape=(1,), name='order_amount')
distance_km = Input(shape=(1,), name='distance_km')

In [159]:
features = Concatenate(name='order_features')([order_amount, distance_km])

In [160]:
x = Dense(32, activation='relu')(features)
x = Dense(16, activation='relu')(x)
time_output = Dense(1, activation='linear', name='delivery_time_mins')(x)
rating_output = Dense(1, activation='relu', name='delivery_rating_stars')(x)

In [161]:
zomato_model = Model(
    inputs=[order_amount, distance_km],
    outputs=[time_output, rating_output],
    name='zomato_delivery_predictor'
)

In [162]:
zomato_model.compile(
    optimizer='adam',
    loss={
        'delivery_time_mins': 'mse',
        'delivery_rating_stars': 'mae'
        },
    loss_weights={
        'delivery_time_mins': 1.0,
        'delivery_rating_stars': 0.5
    }
)

In [163]:
zomato_model.summary()

Model: "zomato_delivery_predictor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ order_amount        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ distance_km         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ order_features      │ (None, 2)         │          0 │ order_amount[0][… │
│ (Concatenate)       │                   │            │ distance_km[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_48 (Dense)    │ (None, 32)        │         96 │ order_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_49 (Dense)    │ (None, 16)        │        528 │ dense_48[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delivery_time_mins  │ (None, 1)         │         17 │ dense_49[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delivery_rating_st… │ (None, 1)         │         17 │ dense_49[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 658 (2.57 KB)

 Trainable params: 658 (2.57 KB)

 Non-trainable params: 0 (0.00 B)

* 3. Modify an existing Keras Sequential model for music playlist recommendations to use the Functional API instead, allowing for future flexibility such as adding more inputs or outputs.<br><br><em><strong>Hint:</strong> Focus on rewriting the model definition, not the data pipeline.</em>

In [164]:
# Before (Sequential)
# model = Sequential([
#     Dense(64, activation='relu', input_shape=(10,)),
#     Dense(32, activation='relu'),
#     Dense(1, activation='sigmoid')
# ])

In [165]:
playlist_features_input = Input(shape=(10,), name='playlist_audio_features')

In [166]:
x = Dense(64, activation='relu', name='dense_1')(playlist_features_input)
x = Dense(32, activation='relu', name='dense_2')(x)
playlist_recommendation_output = Dense(1, activation='sigmoid', name='recommendation_score')(x)

In [167]:
music_model = Model(
    inputs=playlist_features_input,
    outputs=playlist_recommendation_output,
    name='music_playlist_recommender'
)

In [168]:
music_model.summary()

Model: "music_playlist_recommender"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ playlist_audio_features         │ (None, 10)             │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ recommendation_score (Dense)    │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

* 4. Implement a skip connection in a Keras Functional API model for an IPL cricket score predictor, where the input layer connects both to an intermediate Dense layer and directly to the output layer.<br><br><em><strong>Hint:</strong> Use the 'Add' layer to combine outputs from different layers.</em>

In [169]:
from tensorflow.keras.layers import Add

In [170]:
ipl_input = Input(shape=(8,), name='match_context_features')

In [171]:
x = Dense(32, activation='relu')(ipl_input)
x = Dense(8, activation='relu', name='intermediate_dense')(x)
skip_combined = Add(name='residual_skip_connection')([ipl_input, x])

In [172]:
y = Dense(16, activation='relu')(skip_combined)
predicted_score = Dense(1, activation='linear', name='final_projected_score')(y)

In [173]:
ipl_model = Model(
    inputs=ipl_input,
    outputs=predicted_score,
    name='ipl_score_predictor'
)

In [174]:
ipl_model.summary()

Model: "ipl_score_predictor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ match_context_feat… │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_50 (Dense)    │ (None, 32)        │        288 │ match_context_fe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ intermediate_dense  │ (None, 8)         │        264 │ dense_50[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_skip_conn… │ (None, 8)         │          0 │ match_context_fe… │
│ (Add)               │                   │            │ intermediate_den… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_51 (Dense)    │ (None, 16)        │        144 │ residual_skip_co… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_projected_sc… │ (None, 1)         │         17 │ dense_51[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 713 (2.79 KB)

 Trainable params: 713 (2.79 KB)

 Non-trainable params: 0 (0.00 B)